<a href="https://colab.research.google.com/github/Aswathi281099/Generative-Artificial-Intelligence/blob/main/AI_TASK_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TASK 3**

In [1]:
!pip install -U bitsandbytes accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 74.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [8]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from typing import Dict, List, Any

class Llama3LayerLogitExtractor:
    """
    Extracts intermediate hidden states, layer-wise logits, and sequence entropy
    from 4-bit quantized models (Qwen, Llama, Mistral).
    """
    def __init__(self, model_id: str = "Qwen/Qwen2.5-7B-Instruct"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True
        )

        print(f"Loading 4-bit Quantized Model: {model_id}...")
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto",
            dtype=torch.bfloat16
        )
        self.model.eval()

        self.layer_activations: Dict[int, torch.Tensor] = {}
        self.hooks = []
        self._register_hooks()

    def _register_hooks(self):
        """Attaches forward hooks across model layers."""
        layers = getattr(self.model.model, "layers", None)
        if layers is None:
            raise AttributeError("Could not locate decoder layers in model.model.layers")

        for layer_idx, layer in enumerate(layers):
            def create_hook(idx: int):
                def hook(module, input_tensor, output_tensor):
                    # Handle tuple vs single tensor return
                    hidden_state = output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor
                    self.layer_activations[idx] = hidden_state.detach()
                return hook

            h = layer.register_forward_hook(create_hook(layer_idx))
            self.hooks.append(h)

    def calculate_entropy(self, logits: torch.Tensor) -> float:
        probs = F.softmax(logits, dim=-1)
        log_probs = F.log_softmax(logits, dim=-1)
        return -torch.sum(probs * log_probs, dim=-1).item()

    def extract_layer_wise_logits(self, prompt: str) -> Dict[str, Any]:
        self.layer_activations.clear()
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            final_outputs = self.model(**inputs)
            final_logits = final_outputs.logits[0, -1, :]

        lm_head = self.model.lm_head
        final_norm = self.model.model.norm

        layer_results = []
        for layer_idx in sorted(self.layer_activations.keys()):
            activation = self.layer_activations[layer_idx]

            # --- FIX: Dynamic Shape Handling for 2D vs 3D Tensors ---
            if activation.dim() == 3:
                # Shape: (batch_size, seq_len, hidden_dim)
                layer_hidden = activation[0, -1:, :]  # Keep (1, hidden_dim)
            elif activation.dim() == 2:
                # Shape: (seq_len, hidden_dim)
                layer_hidden = activation[-1:, :]     # Keep (1, hidden_dim)
            else:
                raise ValueError(f"Unexpected tensor dimension: {activation.dim()}")

            # Pass through Final LayerNorm and LM Head
            normed_hidden = final_norm(layer_hidden)
            layer_logits = lm_head(normed_hidden).view(-1) # Flatten to (vocab_size,)

            entropy = self.calculate_entropy(layer_logits)
            topk_probs, topk_indices = torch.topk(F.softmax(layer_logits, dim=-1), k=5)
            topk_tokens = [
                (self.tokenizer.decode([idx.item()]), prob.item())
                for idx, prob in zip(topk_indices, topk_probs)
            ]

            layer_results.append({
                "layer": layer_idx,
                "entropy": entropy,
                "top_1_token": topk_tokens[0][0],
                "top_1_prob": topk_tokens[0][1],
                "top_5": topk_tokens
            })

        return {
            "prompt": prompt,
            "final_logits": final_logits,
            "layer_results": layer_results
        }

    def cleanup(self):
        for h in self.hooks:
            h.remove()

In [9]:
# Initialize with the already-cached Qwen model weights
extractor = Llama3LayerLogitExtractor(model_id="Qwen/Qwen2.5-7B-Instruct")

prompt_text = "The capital of France is"
analysis = extractor.extract_layer_wise_logits(prompt_text)

print(f"\n--- Layer-wise Trajectory for Prompt: '{prompt_text}' ---")
print(f"{'Layer':<6} | {'Entropy (nats)':<14} | {'Top-1 Token':<18} | {'Probability':<10}")
print("-" * 60)

for res in analysis["layer_results"]:
    # Sample every 4th layer for readability
    if res["layer"] % 4 == 0 or res["layer"] == len(analysis["layer_results"]) - 1:
        print(f"L{res['layer']:<5} | {res['entropy']:<14.4f} | '{res['top_1_token']}':<18 | {res['top_1_prob']:<10.4f}")

extractor.cleanup()

Loading 4-bit Quantized Model: Qwen/Qwen2.5-7B-Instruct...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


--- Layer-wise Trajectory for Prompt: 'The capital of France is' ---
Layer  | Entropy (nats) | Top-1 Token        | Probability
------------------------------------------------------------
L0     | 5.4688         | '从根本':<18 | 0.3867    
L4     | 2.0625         | '从根本':<18 | 0.7305    
L8     | 1.1875         | ' 生命周期':<18 | 0.8008    
L12    | 3.3438         | ' 自动生成':<18 | 0.4316    
L16    | 2.5156         | '℠':<18 | 0.4141    
L20    | 5.2500         | ' strugg':<18 | 0.2412    
L24    | 0.7852         | ' Paris':<18 | 0.8125    
L27    | 2.8125         | ' Paris':<18 | 0.4414    
